In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
from PIL import Image
import requests
import torch
from cotracker.predictor import CoTrackerPredictor
import copy
import h5py
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# load the processor
processor = AutoProcessor.from_pretrained(
    "allenai/Molmo-7B-D-0924",
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
)

# load the model
model = AutoModelForCausalLM.from_pretrained(
    "allenai/Molmo-7B-D-0924",
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
)

cotracker_ckpt_file = (
    "/scr/matthewh6/robot_learning/robot_learning/data/optical_flow/scaled_offline.pth"
)
cotracker = CoTrackerPredictor(checkpoint=cotracker_ckpt_file)
cotracker = cotracker.to("cuda")

path = "/tmp/square_500/square/demo_src_square_task_D0_r_Panda/demo.hdf5"

with h5py.File(path, "a") as f:  # open in append mode so we can add new data
    demo_keys = list(f["data"].keys())
    demo_keys_sorted = sorted(demo_keys, key=lambda k: int(k.split("_")[-1]))

    num_demos = 0
    for demo_key in demo_keys_sorted:
        # print last couple of actions
        video = f["data"][demo_key]["obs"]["agentview_image"][:]
        init_frame = video[0]
        init_frame = Image.fromarray(init_frame, "RGB")

        # process the image and text
        inputs = processor.process(
            images=[init_frame], text="Point to the center of the square-shaped block with a handle."
        )

        # move inputs to the correct device and make a batch of size 1
        inputs = {k: v.to(model.device).unsqueeze(0) for k, v in inputs.items()}

        # generate output; maximum 200 new tokens; stop generation when <|endoftext|> is generated
        output = model.generate_from_batch(
            inputs,
            GenerationConfig(max_new_tokens=200, stop_strings="<|endoftext|>"),
            tokenizer=processor.tokenizer,
        )

        # only get generated tokens; decode them to text
        generated_tokens = output[0, inputs["input_ids"].size(1) :]
        generated_text = processor.tokenizer.decode(
            generated_tokens, skip_special_tokens=True
        )

        print(generated_text)

        # parse the generated text
        parsed_points = parse_points(generated_text)
        print(parsed_points)

        # scale the points
        scaled_parsed_points = scale_points_to_image(parsed_points, init_frame.size)
        print(scaled_parsed_points)

        draw_points_on_image(
            copy.deepcopy(init_frame), scaled_parsed_points, color=(255, 0, 0)
        )

        scaled_parsed_points = np.array(scaled_parsed_points)
        scaled_parsed_points = np.concatenate(
            [np.zeros((scaled_parsed_points.shape[0], 1)), scaled_parsed_points], axis=1
        )

        pred_points, pred_visibility = cotracker(
            torch.tensor(video).permute(0, 3, 1, 2)[None].float().to("cuda"),
            queries=torch.tensor(scaled_parsed_points[None]).float().to("cuda"),
            backward_tracking=True,
        )

        pred_points = pred_points[0].squeeze(1).cpu().numpy()

        f["data"][demo_key].create_dataset("goal_points", data=pred_points)
        print(f"Saved pred_points to {demo_key}/goal_points")